# Filling in gaps in NHNS-J Mean Steps (2015–2023)

Author: Bram Schonfeldt

This notebook assesses using Fitbit steps to fill in gaps between national estimates. We introduce gaps in NHNS-J data where we have access to ground-truth national estimates.

Metrics: MAE, MPD (%), and MAPE (%)

All models are evaluated across three primary metrics:

1. Census-Weighted Mean Absolute Error (MAE, steps/day): sum over k from 1 to 12 of w_k * |Y_hat_k - Y_k|.
2. Census-Weighted Marginal Percentage Discrepancy (MPD, %): sum over k from 1 to 12 of w_k * ((Y_hat_k - Y_k) / Y_k) * 100%.
3. Census-Weighted Mean Absolute Percentage Error (MAPE, %): sum over k from 1 to 12 of w_k * |(Y_hat_k - Y_k) / Y_k| * 100%.

Evaluation protocol:

- For every held-out target survey year t_target in [2016, 2017, 2018, 2019, 2022, 2023], candidate models are trained on all data visible up to each historical base year t_base < t_target (spanning forecasting horizons h = 1 to 8 years). Shorter horizons naturally contribute more observed calendar transitions (h=1: t=5; h=2,3,4: t=3; h=5,6,7: t=2; h=8: t=1).
- Empirical uncertainty is captured via:
  - Median and Interquartile Range (Q1–Q3; 25th–75th percentiles) for the primary headline table and age breakdown table (N = 21 transitions).
  - Median (Min–Max) alongside test transition counts (t) for the multi-horizon breakdown table and Figure 2 error whiskers across lead times h = 1 to 8.

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os
import pickle as pkl
import sys
from typing import Any, Dict, List, Optional, Tuple

from IPython.display import HTML, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Set rcParams
plt.rcdefaults()
plt.rcParams['figure.dpi'] = 200

sys.path.insert(0, os.path.abspath('../..'))
import stratify as strat

Load in the data...

In [ ]:
internal_dir = strat.get_internal_data_dir()
public_dir = strat.get_public_data_dir()
START_DATE = '2015-01-01'
END_DATE = '2026-01-01'
N_SHARDS = 100

UN_POP_MALE_PATH = (
    f'{public_dir}/WPP2024_POP_F02_2_POPULATION_5-YEAR_AGE_GROUPS_MALE.xlsx'
)
UN_POP_FEMALE_PATH = (
    f'{public_dir}/WPP2024_POP_F02_3_POPULATION_5-YEAR_AGE_GROUPS_FEMALE.xlsx'
)

# 1. Load pre-computed Fitbit Japan step grid (Nov Mon-Sat matched).
# 9f4397029d8039e43903bb5b8b7e75ec: (target_country='Japan',
# min_days_per_user=1, min_age=20, max_age=104, min_steps=100, max_steps=50000,
# start_month=11, stop_month=11, exclude_days_of_week=[6],
# exclude_outliers=False, exclude_extreme_values=False).
step_grid = strat.StratifySteps.from_shards(
    path_base=internal_dir,
    start_date=START_DATE,
    end_date=END_DATE,
    n_shards=N_SHARDS,
    un_pop_male_path=UN_POP_MALE_PATH,
    un_pop_female_path=UN_POP_FEMALE_PATH,
    min_days_per_user=1,
    min_age=20,
    max_age=104,
    min_steps=100,
    max_steps=50000,
    target_country='Japan',
    start_month=11,
    stop_month=11,
    exclude_days_of_week=[6],
    override_hash='9f4397029d8039e43903bb5b8b7e75ec',
)

# 2. Load Japan NHNS-J processed dataset (2015-2023)
japan_step_path = f'{public_dir}/2015_2024_japan_steps.pkl'
if strat.file_exists(japan_step_path):
  japan_steps_raw = strat.load_pickle(japan_step_path)
  print(f'Loaded NHNS-J step data: {len(japan_steps_raw)} rows.')
else:
  raise FileNotFoundError('NHNS-J processed dataset not found.')

Data processing:
- Map age to compatible 10-year groups,
- Compute mean and standard deviation of Fitbit steps,
- Get UN population weights,


In [ ]:
"""Data cleaning, stratum aggregation, and fixed demographic census weighting."""

# Free-living linear calibration constants mapping wrist-worn Fitbit step counts
# to waist-worn Yamax pedometer equivalents (Fitbit = 4152.0 + 0.8 * Yamax),
# from Nakagata et al. (2022), Gait & Posture, 98, 24-33,
# https://doi.org/10.1016/j.gaitpost.2022.08.004.
NAKAGATA_INTERCEPT = 4152.0
NAKAGATA_SLOPE = 0.8


def assign_midage_10(age: int) -> int:
  """Maps individual age to 10-year midpoint bracket representing survey strata.

  Args:
      age: Integer age of participant.

  Returns:
      Midpoint integer (25 for 20-29, 35 for 30-39, ..., 75 for 70+).
  """
  if age < 70:
    return age // 10 * 10 + 5
  return 75  # 75 represents 70+ bracket


# 1. Process Fitbit step_grid for Japan
fb_df = step_grid.results_grid.reset_index()
fb_df = fb_df[fb_df['country'] == 'Japan'].copy()
fb_df['midage_10'] = fb_df['midage'].apply(assign_midage_10)

fb_df = (
    fb_df.groupby(['year', 'gender', 'midage_10'])[
        ['steps', 'steps_sq', 'readings']
    ]
    .sum()
    .reset_index()
)
fb_df['gender'] = fb_df['gender'].replace({0: 'male', 1: 'female'})

fb_step_mean = fb_df.copy()
fb_step_mean['mean_steps_fitbit'] = (
    fb_step_mean['steps'] / fb_step_mean['readings']
)
fb_step_mean['var_steps_fitbit'] = (
    fb_step_mean['steps_sq'] / fb_step_mean['readings']
    - fb_step_mean['mean_steps_fitbit'] ** 2
)
fb_step_mean['std_steps_fitbit'] = np.sqrt(fb_step_mean['var_steps_fitbit'])
fb_step_mean = fb_step_mean.rename(columns={'readings': 'n_fitbit'})

# UN population counts for Japan
japan_pop_count = step_grid.un_pop[
    step_grid.un_pop['country'] == 'Japan'
].copy()
japan_pop_count['midage_10'] = japan_pop_count['age'].apply(assign_midage_10)
japan_pop_count['gender'] = japan_pop_count['gender'].replace(
    {0: 'male', 1: 'female'}
)
japan_pop_count = (
    japan_pop_count.groupby(['year', 'gender', 'midage_10'])['Count']
    .sum()
    .reset_index()
)
# UN WPP population counts are reported in thousands
japan_pop_count['n_un'] = japan_pop_count['Count'] * 1000

fb_step_mean = fb_step_mean.merge(
    japan_pop_count[['year', 'gender', 'midage_10', 'n_un']],
    on=['year', 'gender', 'midage_10'],
    how='inner',
)

# 2. Process Japanese NHNS-J steps
japan_steps = japan_steps_raw.copy()
age_dict = {
    '20-29': 25,
    '30-39': 35,
    '40-49': 45,
    '50-59': 55,
    '60-69': 65,
    '70+': 75,
}
japan_steps = japan_steps[japan_steps['age_group'].isin(age_dict.keys())].copy()
japan_steps['midage_10'] = japan_steps['age_group'].map(age_dict)
japan_steps = japan_steps.rename(
    columns={
        'mean_steps': 'mean_steps_nhnsj',
        'std_steps': 'std_steps_nhnsj',
        'n_participants': 'n_nhnsj',
    }
)

# 3. Merge into consolidated japan_estimates dataframe
fb_cols = [
    'year',
    'gender',
    'midage_10',
    'mean_steps_fitbit',
    'std_steps_fitbit',
    'n_fitbit',
    'n_un',
]
japan_cols = [
    'year',
    'gender',
    'midage_10',
    'mean_steps_nhnsj',
    'std_steps_nhnsj',
    'n_nhnsj',
]
japan_estimates = fb_step_mean[fb_cols].merge(
    japan_steps[japan_cols], on=['year', 'gender', 'midage_10'], how='inner'
)

# Filter valid study years (2015-2023, excluding 2020-2021 COVID disruption waves)
df_jp_clean = japan_estimates[
    (japan_estimates['gender'] != 'all')
    & (japan_estimates['year'] <= 2023)
    & (~japan_estimates['year'].isin([2020, 2021]))
].copy()

age_label_map = {
    25: '20-29',
    35: '30-39',
    45: '40-49',
    55: '50-59',
    65: '60-69',
    75: '70+',
}
df_jp_clean['age_group'] = df_jp_clean['midage_10'].map(age_label_map)
df_jp_clean['gender_label'] = df_jp_clean['gender'].map(
    {'male': 'Men', 'female': 'Women'}
)
df_jp_clean['gender_num'] = (df_jp_clean['gender'] == 'female').astype(float)
df_jp_clean['age_norm'] = df_jp_clean['midage_10'] / 100.0
df_jp_clean['mean_steps_fitbit_naka'] = (
    df_jp_clean['mean_steps_fitbit'] - NAKAGATA_INTERCEPT
) / NAKAGATA_SLOPE

# Fixed multi-year UN census population distribution over 2015-2023 (Japan)
jp_stratum_pop = (
    df_jp_clean.groupby(['age_group', 'gender_label'])['n_un']
    .mean()
    .reset_index()
)
jp_total_pop = jp_stratum_pop['n_un'].sum()
jp_stratum_pop['w_un_pop'] = jp_stratum_pop['n_un'] / jp_total_pop
jp_stratum_pop['w_grid'] = 1.0 / 12.0

df_jp_clean = df_jp_clean.merge(
    jp_stratum_pop[['age_group', 'gender_label', 'w_un_pop', 'w_grid']],
    on=['age_group', 'gender_label'],
)

# Establish historical previous-wave mapping for benchmark
years_sorted = sorted(df_jp_clean['year'].unique())
prev_year_map = {
    years_sorted[i]: years_sorted[i - 1] for i in range(1, len(years_sorted))
}
df_jp_clean['prev_year'] = df_jp_clean['year'].map(prev_year_map)

print(
    f'Japan cleaned evaluation rows: {len(df_jp_clean)} stratum-year cells'
    f' ({len(years_sorted)} years x 12 strata).'
)
print(f'Evaluation Survey Waves: {years_sorted}')
print(f'Previous Wave Transitions for Benchmark: {prev_year_map}')
print('Data processing and fixed census weights established.')

Candidate models:
1. `M0_Prev_Year_Benchmark`: Uses the last available national estimate as the
estimate for the current year.
2. `M1_Raw_Fitbit`: Uses the current unadjusted Fitbit estimate as the national
estimate,
3. `M2_Nakagata_Fixed`: Applies the adjustment, Y_hat = (F - 4152)/0.8, from
Nakagata et al. to the Fitbit data and uses this as the estimate,
4. `M3_OLS_Demo_Steps_Interaction`: OLS regression based on Fitbit estimates,
but with age and sex: Y_hat = b_0 + b_1xF + b_2xA + b_3xS + b_4xAxS
5. `M4_Exponential_Demo_Discount`: Non-linear model which stratifies by sex,
and models steps with an exponential discount function of age. For each sex:
Y_hat/F = b_0  + b_1 exp( b_2 A )

In [ ]:
class ForwardModelRegistry:
  """Registers and executes candidate calibration models on train/test historical windows."""

  MODEL_DEFS = {
      'M0_Prev_Year_Benchmark': (
          'Previous Year Census/Survey Estimate (Baseline)',
          0,
      ),
      'M1_Raw_Fitbit': ('Unadjusted Fitbit Prediction', 0),
      'M2_Nakagata_Fixed': ('Fitbit + Nakagata Adjustment', 0),
      'M3_OLS_Demo_Steps_Interaction': (
          'Fitbit + Age + Sex + Age*Sex OLS',
          5,
      ),
      'M4_Exponential_Demo_Discount': (
          'Sex-Stratified Exponential Model',
          6,
      ),
  }

  @staticmethod
  def fit_and_predict(
      model_key: str,
      train_df: pd.DataFrame,
      test_df: pd.DataFrame,
      full_history_df: pd.DataFrame,
  ) -> Tuple[pd.Series, Any]:
    """Fits specified model on train_df (years < t) and predicts on test_df (year t).

    Args:
        model_key: Key name of the model in MODEL_DEFS.
        train_df: Historical training slice (all survey waves strictly before
          test year).
        test_df: Out-of-sample evaluation slice (current test year t).
        full_history_df: Full cleaned dataset used for benchmark previous-wave
          lookups.

    Returns:
        Tuple of (predicted_series, fitted_model_object).
    """
    preds = pd.Series(index=test_df.index, dtype=float)
    fit_obj = None

    tr = train_df.copy()
    te = test_df.copy()
    for d in (tr, te):
      d['GA'] = d['age_norm'] * d['gender_num']

    if model_key == 'M0_Prev_Year_Benchmark':
      for idx, row in test_df.iterrows():
        y_prev = row['prev_year']
        if pd.isna(y_prev):
          preds.loc[idx] = np.nan
        else:
          match = full_history_df[
              (full_history_df['year'] == y_prev)
              & (full_history_df['gender'] == row['gender'])
              & (full_history_df['midage_10'] == row['midage_10'])
          ]
          if not match.empty:
            preds.loc[idx] = match['mean_steps_nhnsj'].values[0]
          else:
            preds.loc[idx] = np.nan

    elif model_key == 'M1_Raw_Fitbit':
      preds = test_df['mean_steps_fitbit'].copy()

    elif model_key == 'M2_Nakagata_Fixed':
      preds = (
          test_df['mean_steps_fitbit'] - NAKAGATA_INTERCEPT
      ) / NAKAGATA_SLOPE

    elif model_key == 'M3_OLS_Demo_Steps_Interaction':
      cols = ['mean_steps_fitbit', 'age_norm', 'gender_num', 'GA']
      X_tr = sm.add_constant(tr[cols], has_constant='add')
      y_tr = tr['mean_steps_nhnsj']
      fit_obj = sm.OLS(y_tr, X_tr).fit()

      X_te = sm.add_constant(te[cols], has_constant='add')
      preds.loc[test_df.index] = fit_obj.predict(X_te)

    elif model_key == 'M4_Exponential_Demo_Discount':

      def age_fn(age, a, b, c):
        return a + b * np.exp(age * c)

      init_p = [1.0, -0.3, 1.0]
      fit_obj = {}
      tr['ratio'] = tr['mean_steps_nhnsj'] / tr['mean_steps_fitbit']

      for gender_val in ['male', 'female']:
        tr_sub = tr[tr['gender'] == gender_val].dropna(
            subset=['ratio', 'age_norm']
        )
        te_sub = te[te['gender'] == gender_val]
        if len(tr_sub) >= 3:
          try:
            popt, _ = curve_fit(
                age_fn,
                tr_sub['age_norm'],
                tr_sub['ratio'],
                p0=init_p,
                maxfev=10000,
            )
          except Exception:
            popt = np.array(init_p)
          fit_obj[gender_val] = popt
          if not te_sub.empty:
            ratio_hat = age_fn(te_sub['age_norm'], *popt)
            preds.loc[te_sub.index] = ratio_hat * te_sub['mean_steps_fitbit']
        else:
          preds.loc[te_sub.index] = te_sub['mean_steps_fitbit_naka']

    else:
      raise ValueError(f'Unknown model key: {model_key}')

    return preds, fit_obj


print('ForwardModelRegistry initialized with 5 streamlined models.')

### Multi-Lag Evaluation Protocol
1. **All Valid Historical Transitions (21 Transition Pairs)**:
   For every held-out target survey year $t_{\text{target}} \in [2016, 2017, 2018, 2019, 2022, 2023]$, candidate models are trained on all data visible up to each historical base year $t_{\text{base}} < t_{\text{target}}$ (covering all positive prediction gaps $h = t_{\text{target}} - t_{\text{base}} \in \{1, 2, 3, 4, 5, 6, 7, 8\}$ years).
2. **Census-Weighted Stratum Aggregation**:
   For each individual transition pair $p$, out-of-sample predictions across the 12 age-sex demographic strata are evaluated using fixed UN population weights $w_k$.
3. **Distributional Uncertainty Representation**:
   Because sample sizes vary across lead-time horizons ($t=5$ for $h=1$ down to $t=1$ for $h=8$) and error metrics are non-negative and skewed:
   - The headline evaluation and age breakdown report the **Median and Interquartile Range (Q1–Q3; 25th–75th percentiles)** across all 21 test transitions.
   - The multi-horizon evaluation reports the **Median [Min–Max] along with the exact number of test transitions ($t$)** for each horizon $h \in \{1\dots 8\}$.

In [ ]:
class MultiLagEvaluator:
  """Evaluates models across all 21 valid historical transitions (1-8 years)."""

  def __init__(self, df: pd.DataFrame):
    self.df = df.copy().sort_values(['year', 'gender', 'midage_10'])
    self.all_years = sorted(self.df['year'].unique())
    self.target_years = self.all_years[1:]  # 2016, 2017, 2018, 2019, 2022, 2023
    self.registry = ForwardModelRegistry()

  def run_evaluation(
      self,
  ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Runs evaluation across all 21 historical transition pairs."""
    records = []
    models_eval = list(self.registry.MODEL_DEFS.keys())

    for t_target in self.target_years:
      valid_bases = [t for t in self.all_years if t < t_target]
      test_slice = self.df[self.df['year'] == t_target].copy()

      for t_base in valid_bases:
        h = t_target - t_base
        train_slice = self.df[self.df['year'] <= t_base].copy()
        base_slice = self.df[self.df['year'] == t_base].set_index(
            ['gender', 'midage_10']
        )['mean_steps_nhnsj']

        for model_key in models_eval:
          if model_key == 'M0_Prev_Year_Benchmark':
            for idx, row in test_slice.iterrows():
              y_act = row['mean_steps_nhnsj']
              y_hat = base_slice.loc[(row['gender'], row['midage_10'])]
              err = y_hat - y_act
              pct_disc = (
                  ((y_hat - y_act) / y_act) * 100.0 if y_act != 0 else np.nan
              )
              records.append({
                  't_target': t_target,
                  't_base': t_base,
                  'horizon': h,
                  'transition': f'{t_base}->{t_target}',
                  'model_key': model_key,
                  'gender': row['gender'],
                  'gender_label': row['gender_label'],
                  'age_group': row['age_group'],
                  'midage_10': row['midage_10'],
                  'w_un_pop': row['w_un_pop'],
                  'actual_nhnsj': y_act,
                  'pred_nhnsj': y_hat,
                  'error': err,
                  'abs_error': np.abs(err),
                  'pct_discrepancy': pct_disc,
                  'abs_pct_error': np.abs(pct_disc),
              })
          else:
            preds, _ = self.registry.fit_and_predict(
                model_key=model_key,
                train_df=train_slice,
                test_df=test_slice,
                full_history_df=self.df,
            )
            for idx, row in test_slice.iterrows():
              y_act = row['mean_steps_nhnsj']
              y_hat = preds.loc[idx]
              err = y_hat - y_act
              pct_disc = (
                  ((y_hat - y_act) / y_act) * 100.0 if y_act != 0 else np.nan
              )
              records.append({
                  't_target': t_target,
                  't_base': t_base,
                  'horizon': h,
                  'transition': f'{t_base}->{t_target}',
                  'model_key': model_key,
                  'gender': row['gender'],
                  'gender_label': row['gender_label'],
                  'age_group': row['age_group'],
                  'midage_10': row['midage_10'],
                  'w_un_pop': row['w_un_pop'],
                  'actual_nhnsj': y_act,
                  'pred_nhnsj': y_hat,
                  'error': err,
                  'abs_error': np.abs(err),
                  'pct_discrepancy': pct_disc,
                  'abs_pct_error': np.abs(pct_disc),
              })

    df_preds_all = pd.DataFrame(records)

    # 1. Compute Census-Weighted metrics for each individual transition pair (N = 21 pairs)
    trans_records = []
    for (t_target, t_base, model_key), sub_df in df_preds_all.groupby(
        ['t_target', 't_base', 'model_key']
    ):
      w_norm = sub_df['w_un_pop'] / sub_df['w_un_pop'].sum()
      mae_tr = np.sum(sub_df['abs_error'] * w_norm)
      mpd_tr = np.sum(sub_df['pct_discrepancy'] * w_norm)
      mape_tr = np.sum(sub_df['abs_pct_error'] * w_norm)
      trans_records.append({
          't_target': t_target,
          't_base': t_base,
          'horizon': t_target - t_base,
          'transition': f'{t_base}->{t_target}',
          'model_key': model_key,
          'mae': mae_tr,
          'mpd': mpd_tr,
          'mape': mape_tr,
      })
    df_trans = pd.DataFrame(trans_records)

    # 2. Headline Summary (Method 1: Pooled over all 21 transition pairs)
    headline_rows = []
    bench_trans_mae = df_trans[
        df_trans['model_key'] == 'M0_Prev_Year_Benchmark'
    ].set_index('transition')['mae']
    bench_mean_mae = bench_trans_mae.mean()

    for model_key, (desc, _) in self.registry.MODEL_DEFS.items():
      sub = df_trans[df_trans['model_key'] == model_key]
      maes = sub['mae'].values
      mpds = sub['mpd'].values
      mapes = sub['mape'].values

      mean_mae = np.mean(maes)
      med_mae = np.median(maes)
      q1_mae, q3_mae = np.percentile(maes, [25, 75])

      mean_mpd = np.mean(mpds)
      med_mpd = np.median(mpds)
      q1_mpd, q3_mpd = np.percentile(mpds, [25, 75])

      mean_mape = np.mean(mapes)
      med_mape = np.median(mapes)
      q1_mape, q3_mape = np.percentile(mapes, [25, 75])

      delta_mae = mean_mae - bench_mean_mae

      headline_rows.append({
          'Model Key': model_key,
          'Model Description': desc,
          'Mean MAE': mean_mae,
          'Median MAE': med_mae,
          'MAE (steps/day, Median [Q1--Q3])': (
              f'{med_mae:,.1f} ({q1_mae:,.1f}--{q3_mae:,.1f})'
          ),
          'Mean MPD (%)': mean_mpd,
          'Median MPD (%)': med_mpd,
          'MPD (%, Median [Q1--Q3])': (
              f'{med_mpd:.1f} ({q1_mpd:.1f}--{q3_mpd:.1f})'
          ),
          'Mean MAPE (%)': mean_mape,
          'Median MAPE (%)': med_mape,
          'MAPE (%, Median [Q1--Q3])': (
              f'{med_mape:.1f} ({q1_mape:.1f}--{q3_mape:.1f})'
          ),
          'Delta MAE vs Benchmark': delta_mae,
      })
    df_headline = pd.DataFrame(headline_rows)

    # 3. Horizon / Lag Summary (Median, Min, Max across transition years for each lag)
    lag_rows = []
    for h in sorted(df_trans['horizon'].unique()):
      sub_h = df_trans[df_trans['horizon'] == h]
      n_years = sub_h['transition'].nunique()
      target_years = sorted(sub_h['t_target'].unique())
      target_years_str = ', '.join([str(y) for y in target_years])

      for model_key, (desc, _) in self.registry.MODEL_DEFS.items():
        sub_m = sub_h[sub_h['model_key'] == model_key]
        maes = sub_m['mae'].values
        mpds = sub_m['mpd'].values
        mapes = sub_m['mape'].values

        med_mae = np.median(maes)
        min_mae = np.min(maes)
        max_mae = np.max(maes)

        med_mpd = np.median(mpds)
        min_mpd = np.min(mpds)
        max_mpd = np.max(mpds)

        med_mape = np.median(mapes)
        min_mape = np.min(mapes)
        max_mape = np.max(mapes)

        lag_rows.append({
            'Horizon': f'{h}-Year (h={h})',
            'h': h,
            'Number of Test Years (t)': n_years,
            'Tested Target Years': target_years_str,
            'Model Key': model_key,
            'Model Description': desc,
            'Median MAE': med_mae,
            'Min MAE': min_mae,
            'Max MAE': max_mae,
            'MAE (steps/day, Median [Min--Max])': (
                f'{med_mae:,.1f} ({min_mae:,.1f}--{max_mae:,.1f})'
            ),
            'Median MPD (%)': med_mpd,
            'MPD (%, Median [Min--Max])': (
                f'{med_mpd:.1f} ({min_mpd:.1f}--{max_mpd:.1f})'
            ),
            'Median MAPE (%)': med_mape,
            'Min MAPE': min_mape,
            'Max MAPE': max_mape,
            'MAPE (%, Median [Min--Max])': (
                f'{med_mape:.1f} ({min_mape:.1f}--{max_mape:.1f})'
            ),
        })
    df_lag_summary = pd.DataFrame(lag_rows)

    return df_preds_all, df_trans, df_headline, df_lag_summary


evaluator_multilag = MultiLagEvaluator(df_jp_clean)
df_preds_all, df_trans, df_headline, df_lag_summary = (
    evaluator_multilag.run_evaluation()
)

print(
    'Multi-lag evaluation complete across all 21 historical transition pairs.'
)

Following the Method 1 multi-lag evaluation protocol:
1. **Primary Headline Performance Table**: Pooled evaluation across all 21 historical transition pairs (1–8 year horizons), reporting **Median and Quartiles (Q1–Q3)** for Census-Weighted MAE (steps/day), MPD (%), and MAPE (%).
2. **Pivoted Forecasting Horizon Table**: Accuracy across lead times ($h = 1\dots 8$ years) formatted with candidate models as columns, horizons as rows, and reporting the **Number of Test Years ($t$)** and **Median [Min–Max] MAE (steps/day)**.
3. **Figure 2: Multi-Horizon Forecasting Trajectory Plot**: Trajectory of median error across lead-time horizons $h=1\dots 8$ with **empirical Min–Max error whiskers**.
4. **Pivoted Age Group Breakdown Table**: Census-weighted cohort MAE across the 21 transitions comparing working-age adults (20–50) vs. older adults (50+) and 10-year age brackets (Median [Q1–Q3]).

In [ ]:
# Mapping for concise, standardized model labels across all outputs
model_order = [
    'M0_Prev_Year_Benchmark',
    'M1_Raw_Fitbit',
    'M2_Nakagata_Fixed',
    'M3_OLS_Demo_Steps_Interaction',
    'M4_Exponential_Demo_Discount',
]
model_col_names = {
    'M0_Prev_Year_Benchmark': 'Previous Survey Estimate (Baseline)',
    'M1_Raw_Fitbit': 'Unadjusted Fitbit',
    'M2_Nakagata_Fixed': 'Nakagata et al. (2022)',
    'M3_OLS_Demo_Steps_Interaction': 'Fitbit + Demographics OLS',
    'M4_Exponential_Demo_Discount': 'Exponential Age Discount',
}

# ==========================================================================================
# 1. PRIMARY HEADLINE TABLE: MULTI-LAG PERFORMANCE ACROSS ALL 21 TRANSITIONS
# ==========================================================================================
print('=' * 135)
print(
    '1. PRIMARY HEADLINE TABLE: MULTI-LAG PERFORMANCE POOLED OVER ALL 21'
    ' TRANSITION PAIRS (1-8 YEARS)'
)
print('=' * 135)

df_headline_clean = df_headline.copy()
df_headline_clean['Model / Benchmark'] = df_headline_clean['Model Key'].map(
    model_col_names
)
df_headline_display = df_headline_clean[[
    'Model / Benchmark',
    'MAE (steps/day, Median [Q1--Q3])',
    'MPD (%, Median [Q1--Q3])',
    'MAPE (%, Median [Q1--Q3])',
]]

display(df_headline_display)

# LaTeX Code for Headline Table
latex_headline_lines = [
    r'\begin{table*}[htbp]',
    r'\centering',
    (
        r'\caption{\textbf{Primary evaluation: National step count estimation'
        r' performance using contemporary wearable data calibrated on'
        r' historical survey benchmarks.} Census-weighted Mean Absolute Error'
        r' (MAE, steps/day), Marginal Percentage Discrepancy (MPD, \%), and'
        r' Mean Absolute Percentage Error (MAPE, \%) pooled across all 21'
        r' historical calibration transitions (spanning 1--8 year survey lags).'
        r' Median and interquartile ranges (Q1--Q3) represent empirical'
        r' uncertainty across evaluation years.}'
    ),
    r'\label{tab:multilag_forecasting_headline}',
    r'\begin{tabular}{lccc}',
    r'\toprule',
    (
        r'\textbf{Model / Benchmark} & \textbf{MAE (steps/day, Median'
        r' [Q1--Q3])} & \textbf{MPD (\%, Median [Q1--Q3])} & \textbf{MAPE (\%,'
        r' Median [Q1--Q3])} \\'
    ),
    r'\midrule',
]
for _, r in df_headline_display.iterrows():
  m_name = r['Model / Benchmark'].replace('–', '--')
  mae_str = r['MAE (steps/day, Median [Q1--Q3])'].replace('–', '--')
  mpd_str = r['MPD (%, Median [Q1--Q3])'].replace('–', '--')
  mape_str = r['MAPE (%, Median [Q1--Q3])'].replace('–', '--')
  latex_headline_lines.append(
      f'{m_name} & {mae_str} & {mpd_str} & {mape_str} \\\\'
  )
latex_headline_lines.extend([
    r'\bottomrule',
    r'\end{tabular}',
    r'\end{table*}',
])
print('\n' + '=' * 135)
print('LATEX CODE FOR PRIMARY HEADLINE FORECASTING TABLE:')
print('=' * 135)
print('\n'.join(latex_headline_lines))
print('=' * 135)

# ==========================================================================================
# 2. PIVOTED HORIZON / LAG BREAKDOWN TABLE (COMPETITIVE CALIBRATED MODELS)
# ==========================================================================================
# Dropping raw unadjusted Fitbit to preserve horizontal table space for calibrated models
model_order_horizon = [
    'M0_Prev_Year_Benchmark',
    'M2_Nakagata_Fixed',
    'M3_OLS_Demo_Steps_Interaction',
    'M4_Exponential_Demo_Discount',
]

horizon_rows = []
for h in sorted(df_lag_summary['h'].unique()):
  sub_h = df_lag_summary[df_lag_summary['h'] == h]
  t_val = sub_h['Number of Test Years (t)'].values[0]

  row_dict = {
      'Survey Lag (h)': f'{h}-Year (h={h})',
      'Test Years (t)': t_val,
  }
  for m_k in model_order_horizon:
    sub_m = sub_h[sub_h['Model Key'] == m_k]
    col_name = model_col_names[m_k]
    if not sub_m.empty:
      med_mape = sub_m['Median MAPE (%)'].values[0]
      min_mape = sub_m['Min MAPE'].values[0]
      max_mape = sub_m['Max MAPE'].values[0]
      if t_val == 1 or min_mape == max_mape:
        row_dict[col_name] = f'{med_mape:.1f}'
      else:
        row_dict[col_name] = f'{med_mape:.1f} ({min_mape:.1f}--{max_mape:.1f})'
    else:
      row_dict[col_name] = 'N/A'
  horizon_rows.append(row_dict)

df_horizon_pivot = pd.DataFrame(horizon_rows)

print('\n' + '=' * 135)
print(
    '2. PIVOTED FORECASTING HORIZON TABLE: MAPE (%, Median [Min--Max]) ACROSS'
    ' LAGS (h = 1..8 YEARS)'
)
print('=' * 135)
display(df_horizon_pivot)

# LaTeX Code for Pivoted Horizon Table
latex_horizon_pivot_lines = [
    r'\begin{table*}[htbp]',
    r'\centering',
    r'\setlength{\tabcolsep}{4.5pt}',
    (
        r'\caption{\textbf{Performance estimating national step counts from'
        r' contemporary wearable data calibrated on historical survey'
        r' benchmarks.} Census-weighted Mean Absolute Percentage Error (MAPE,'
        r' \%) across lags of $h = 1\dots 8$ years between the calibration'
        r' survey and the target evaluation year ($t$ indicates the number of'
        r' available evaluation years for each lag). Values represent the'
        r' median across evaluation years, with empirical minimum and maximum'
        r' bounds shown in brackets (single point estimates shown when $t=1$).'
        r' \textit{Survey Baseline} carries forward the historical survey'
        r' estimate from $h$ years ago without wearable data; \textit{Nakagata'
        r' et al. (2022)} applies a fixed free-living pedometer conversion;'
        r' \textit{Fitbit + Demographics} fits an age--sex interaction'
        r' regression; \textit{Exponential Discount} applies a sex-stratified'
        r' exponential age adjustment.}'
    ),
    r'\label{tab:horizon_mae_pivot}',
    r'\begin{tabular}{lccccc}',
    r'\toprule',
    (
        r'\textbf{Lag ($h$)} & \textbf{$t$} & \textbf{Survey Baseline} &'
        r' \shortstack{\textbf{Nakagata et al.}\\\textbf{(2022)}} &'
        r' \shortstack{\textbf{Fitbit +}\\\textbf{Demographics}} &'
        r' \shortstack{\textbf{Exponential}\\\textbf{Discount}} \\'
    ),
    r'\midrule',
]
for _, r in df_horizon_pivot.iterrows():
  h_label = r['Survey Lag (h)'].replace('–', '--')
  t_v = r['Test Years (t)']
  m0_v = str(r['Previous Survey Estimate (Baseline)']).replace('–', '--')
  m2_v = str(r['Nakagata et al. (2022)']).replace('–', '--')
  m3_v = str(r['Fitbit + Demographics OLS']).replace('–', '--')
  m4_v = str(r['Exponential Age Discount']).replace('–', '--')
  latex_horizon_pivot_lines.append(
      f'{h_label} & {t_v} & {m0_v} & {m2_v} & {m3_v} & {m4_v} \\\\'
  )
latex_horizon_pivot_lines.extend([
    r'\bottomrule',
    r'\end{tabular}',
    r'\end{table*}',
])
print('\n' + '=' * 135)
print('LATEX CODE FOR PIVOTED HORIZON TABLE (MAPE %):')
print('=' * 135)
print('\n'.join(latex_horizon_pivot_lines))
print('=' * 135)

# ==========================================================================================
# 3. PLOT OF PERFORMANCE ACROSS ALL TIME HORIZONS (h=1..8) WITHOUT MEDIAN ANNOTATIONS
# ==========================================================================================
PALETTE = {
    'actual': '#0072B2',
    'M0_Prev_Year_Benchmark': '#999999',
    'M3_OLS_Demo_Steps_Interaction': '#D55E00',
    'M4_Exponential_Demo_Discount': '#009E73',
    'M2_Nakagata_Fixed': '#E69F00',
    'M1_Raw_Fitbit': '#CC79A7',
}

# Standard harmonized publication figure size (7.0 x 3.4 inches) matching Notebooks 2-5
fig2, ax2 = plt.subplots(figsize=(7.0, 3.4))
ax2.grid(False)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

horizons_info = df_lag_summary[
    ['h', 'Number of Test Years (t)']
].drop_duplicates()
x_labels_fig2 = []
for _, r in horizons_info.iterrows():
  h_v = r['h']
  w_v = r['Number of Test Years (t)']
  x_labels_fig2.append(f'h={h_v}\nt={w_v}')

x_base = np.arange(len(x_labels_fig2))

# Define clean horizontal offsets (dodging) for four models
models_fig2 = [
    (
        'M0_Prev_Year_Benchmark',
        'Carrying forward estimate from h years ago',
        PALETTE['M0_Prev_Year_Benchmark'],
        's',
        -0.12,
    ),
    (
        'M3_OLS_Demo_Steps_Interaction',
        'Fitbit + age*sex',
        PALETTE['M3_OLS_Demo_Steps_Interaction'],
        'o',
        -0.04,
    ),
    (
        'M4_Exponential_Demo_Discount',
        'Exponential age discount',
        PALETTE['M4_Exponential_Demo_Discount'],
        'D',
        +0.04,
    ),
    (
        'M2_Nakagata_Fixed',
        'Nakagata et al. 2022 adjusted',
        PALETTE['M2_Nakagata_Fixed'],
        'P',
        +0.12,
    ),
]

for model_key, label, color, mk, offset in models_fig2:
  sub_lag = df_lag_summary[df_lag_summary['Model Key'] == model_key]
  y_med = sub_lag['Median MAPE (%)'].values
  y_min = sub_lag['Min MAPE'].values
  y_max = sub_lag['Max MAPE'].values
  yerr_low = np.maximum(0, y_med - y_min)
  yerr_high = np.maximum(0, y_max - y_med)
  yerr = [yerr_low, yerr_high]

  # Plot faint dotted line connecting medians
  ax2.plot(
      x_base + offset,
      y_med,
      linestyle=':',
      linewidth=1.1,
      alpha=0.45,
      color=color,
  )

  # Plot markers and whiskers without solid connecting lines
  ax2.errorbar(
      x_base + offset,
      y_med,
      yerr=yerr,
      label=label,
      color=color,
      marker=mk,
      linestyle='None',
      linewidth=1.2,
      markersize=5.0,
      capsize=2.8,
      capthick=1.1,
      elinewidth=1.1,
      alpha=0.95,
  )

ax2.set_title(
    'Percentage error predicting national step counts from Fitbit estimates'
    ' calibrated h years ago'
)
ax2.set_xticks(x_base)
ax2.set_xticklabels(x_labels_fig2)
ax2.set_xlim(-0.6, len(x_labels_fig2) - 0.4)
ax2.set_xlabel(
    'h = years between calibration survey and evaluation year\nt = number of'
    ' evaluation years',
    labelpad=8,
)
ax2.set_ylabel('Census-weighted\nMAPE (%)', rotation=0, ha='right', labelpad=12)
ax2.legend(frameon=False)
plt.tight_layout()
plt.show()

# ==========================================================================================
# 3B. SIMPLIFIED TWO-MODEL PLOT WITH CENTER ANNOTATIONS AND FAINT CONNECTING LINES
# ==========================================================================================
fig3, ax3 = plt.subplots(figsize=(7.0, 3.4))
ax3.grid(False)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

# Define clean horizontal offsets (dodging) for two-model comparison
models_fig3 = [
    (
        'M0_Prev_Year_Benchmark',
        'Carrying forward estimate from h years ago',
        PALETTE['M0_Prev_Year_Benchmark'],
        's',
        -0.08,
    ),
    (
        'M3_OLS_Demo_Steps_Interaction',
        'Fitbit + age*sex',
        PALETTE['M3_OLS_Demo_Steps_Interaction'],
        'o',
        +0.08,
    ),
]

for model_key, label, color, mk, offset in models_fig3:
  sub_lag = df_lag_summary[df_lag_summary['Model Key'] == model_key]
  y_med = sub_lag['Median MAPE (%)'].values
  y_min = sub_lag['Min MAPE'].values
  y_max = sub_lag['Max MAPE'].values
  yerr_low = np.maximum(0, y_med - y_min)
  yerr_high = np.maximum(0, y_max - y_med)
  yerr = [yerr_low, yerr_high]

  # Plot faint dotted line connecting medians
  ax3.plot(
      x_base + offset,
      y_med,
      linestyle=':',
      linewidth=1.1,
      alpha=0.45,
      color=color,
  )

  # Plot discrete markers and error bars without solid connecting lines
  ax3.errorbar(
      x_base + offset,
      y_med,
      yerr=yerr,
      label=label,
      color=color,
      marker=mk,
      linestyle='None',
      linewidth=1.3,
      markersize=5.2,
      capsize=3.0,
      capthick=1.2,
      elinewidth=1.2,
      alpha=0.95,
  )

  # Label Carry-Forward (`offset < 0`) to the LEFT and Fitbit OLS (`offset > 0`) to the RIGHT
  for idx, y_m in enumerate(y_med):
    is_left = offset < 0
    ax3.annotate(
        f'{y_m:.1f}',
        (x_base[idx] + offset, y_m),
        xytext=(-5, 0) if is_left else (5, 0),
        textcoords='offset points',
        ha='right' if is_left else 'left',
        va='center',
        color=color,
    )

ax3.set_title(
    'Percentage error predicting national step counts from Fitbit estimates'
    ' calibrated h years ago'
)
ax3.set_xticks(x_base)
ax3.set_xticklabels(x_labels_fig2)
ax3.set_xlim(-0.6, len(x_labels_fig2) - 0.4)
ax3.set_xlabel(
    'h = years between calibration survey and evaluation year\nt = number of'
    ' evaluation years',
    labelpad=8,
)
ax3.set_ylabel('Census-weighted\nMAPE (%)', rotation=0, ha='right', labelpad=12)
ax3.legend(frameon=False)
plt.tight_layout()
plt.show()

# ==========================================================================================
# 4. TRANSPOSED AGE GROUP BREAKDOWN TABLE (MODELS AS ROWS, AGE COHORTS AS COLUMNS)
# ==========================================================================================
age_categories = [
    ('20-50 (Working Age)', ['20-29', '30-39', '40-49']),
    ('50+ (Older Adults)', ['50-59', '60-69', '70+']),
    ('20-29', ['20-29']),
    ('30-39', ['30-39']),
    ('40-49', ['40-49']),
    ('50-59', ['50-59']),
    ('60-69', ['60-69']),
    ('70+', ['70+']),
]

unique_transitions = df_preds_all[['t_target', 't_base']].drop_duplicates()

age_transposed_rows = []
for m_k in model_order:
  sub_m = df_preds_all[df_preds_all['model_key'] == m_k]
  row_dict = {
      'Model / Benchmark': model_col_names[m_k],
  }
  for ag_label, ag_list in age_categories:
    trans_mape_list = []
    for _, tr_row in unique_transitions.iterrows():
      t_tar = tr_row['t_target']
      t_bas = tr_row['t_base']
      sub_tr = sub_m[
          (sub_m['t_target'] == t_tar)
          & (sub_m['t_base'] == t_bas)
          & (sub_m['age_group'].isin(ag_list))
      ]
      if not sub_tr.empty:
        w_norm = sub_tr['w_un_pop'] / sub_tr['w_un_pop'].sum()
        cw_mape = np.sum(sub_tr['abs_pct_error'] * w_norm)
        trans_mape_list.append(cw_mape)

    if trans_mape_list:
      med_e = np.median(trans_mape_list)
      q1_e, q3_e = np.percentile(trans_mape_list, [25, 75])
      row_dict[ag_label] = f'{med_e:.1f} ({q1_e:.1f}--{q3_e:.1f})'
    else:
      row_dict[ag_label] = 'N/A'

  age_transposed_rows.append(row_dict)

df_age_transposed = pd.DataFrame(age_transposed_rows)

print('\n' + '=' * 135)
print(
    '4. TRANSPOSED AGE GROUP TABLE: CENSUS-WEIGHTED MAPE (%, Median [Q1--Q3])'
    ' ACROSS 21 TRANSITIONS'
)
print('=' * 135)
display(df_age_transposed)

# LaTeX Code for Compact Primary Cohort Table (20-50 vs 50+)
latex_cohort_compact_lines = [
    r'\begin{table}[htbp]',
    r'\centering',
    (
        r'\caption{\textbf{Calibration accuracy across working-age and older'
        r' adult cohorts.} Census-weighted Mean Absolute Percentage Error'
        r' (MAPE, \%; Median [Q1--Q3]) across working-age (20--50 years) and'
        r' older adult (50+ years) demographic cohorts pooled across all 21'
        r' historical calibration transitions in Japan.}'
    ),
    r'\label{tab:broad_cohort_mae_compact}',
    r'\begin{tabular}{lcc}',
    r'\toprule',
    (
        r'\textbf{Model / Benchmark} & \textbf{20--50 (Working Age)} &'
        r' \textbf{50+ (Older Adults)} \\'
    ),
    r'\midrule',
]
for _, r in df_age_transposed.iterrows():
  m_v = str(r['Model / Benchmark']).replace('–', '--')
  c1_v = str(r['20-50 (Working Age)']).replace('–', '--')
  c2_v = str(r['50+ (Older Adults)']).replace('–', '--')
  latex_cohort_compact_lines.append(f'{m_v} & {c1_v} & {c2_v} \\\\')
latex_cohort_compact_lines.extend([
    r'\bottomrule',
    r'\end{tabular}',
    r'\end{table}',
])
print('\n' + '=' * 135)
print('LATEX CODE FOR PRIMARY BROAD COHORT TABLE (MAPE %, 20-50 vs 50+):')
print('=' * 135)
print('\n'.join(latex_cohort_compact_lines))
print('=' * 135)

# LaTeX Code for Full Transposed Age Table (All 10-Year Brackets)
latex_age_full_lines = [
    r'\begin{table*}[htbp]',
    r'\centering',
    (
        r'\caption{\textbf{Age-stratified calibration accuracy across'
        r' demographic brackets.} Census-weighted Mean Absolute Percentage'
        r' Error (MAPE, \%; Median [Q1--Q3]) across 10-year demographic age'
        r' brackets pooled across all 21 historical calibration transitions in'
        r' Japan.}'
    ),
    r'\label{tab:age_all_brackets_transposed}',
    r'\begin{tabular}{lcccccccc}',
    r'\toprule',
    (
        r'\textbf{Model / Benchmark} & \textbf{20--50} & \textbf{50+} &'
        r' \textbf{20--29} & \textbf{30--39} & \textbf{40--49} &'
        r' \textbf{50--59} & \textbf{60--69} & \textbf{70+} \\'
    ),
    r'\midrule',
]
for _, r in df_age_transposed.iterrows():
  m_v = str(r['Model / Benchmark']).replace('–', '--')
  c1 = str(r['20-50 (Working Age)']).replace('–', '--')
  c2 = str(r['50+ (Older Adults)']).replace('–', '--')
  b1 = str(r['20-29']).replace('–', '--')
  b2 = str(r['30-39']).replace('–', '--')
  b3 = str(r['40-49']).replace('–', '--')
  b4 = str(r['50-59']).replace('–', '--')
  b5 = str(r['60-69']).replace('–', '--')
  b6 = str(r['70+']).replace('–', '--')
  latex_age_full_lines.append(
      f'{m_v} & {c1} & {c2} & {b1} & {b2} & {b3} & {b4} & {b5} & {b6} \\\\'
  )
latex_age_full_lines.extend([
    r'\bottomrule',
    r'\end{tabular}',
    r'\end{table*}',
])
print('\n' + '=' * 135)
print('LATEX CODE FOR FULL TRANSPOSED AGE TABLE (MAPE %, ALL BRACKETS):')
print('=' * 135)
print('\n'.join(latex_age_full_lines))
print('=' * 135)

# *FIN*